In [ ]:
import requests
import sqlite3
import json


def fetch_and_store_universities(country_name, db_path="./data/universities.db"):
    """
    Consulta a Universities API para um país específico e salva os resultados
    em uma tabela SQLite com o nome do país.
    """
    # 1. Obter dados da Universities API
    url = f"http://universities.hipolabs.com/search?country={country_name}"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Erro ao acessar a API: Status Code {response.status_code}")
        return

    universities = response.json()

    if not universities:
        print(f"Nenhuma universidade encontrada para o país: {country_name}")
        return

    print(f"Foram encontradas {len(universities)} universidades em {country_name}.")

    # 2. Conectar ao banco de dados SQLite
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Prepara o nome da tabela (envolvido em aspas duplas caso o país tenha espaços)
    table_name = f'"{country_name}"'

    # 3. Criar a tabela se não existir
    create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT,
            state_province TEXT,
            alpha_two_code TEXT,
            country TEXT,
            domains TEXT,
            web_pages TEXT)
        """

    cursor.execute(create_table_sql)

    # 4. Inserir os dados na tabela
    insert_sql = f"""
        INSERT INTO {table_name} 
        (name, state_province, alpha_two_code, country, domains, web_pages) 
        VALUES (?, ?, ?, ?, ?, ?)
        """

    records_to_insert = []
    for uni in universities:
        # Extrai os campos e serializa as listas (domains e web_pages) para JSON
        records_to_insert.append(
            (
                uni.get("name", ""),
                uni.get("state-province", ""),
                uni.get("alpha_two_code", ""),
                uni.get("country", ""),
                json.dumps(uni.get("domains", [])),
                json.dumps(uni.get("web_pages", [])),
            )
        )

    cursor.executemany(insert_sql, records_to_insert)

    # 5. Efetivar as alterações e fechar a conexão
    conn.commit()
    conn.close()
    print(
        f"Dados salvos com sucesso na tabela {table_name} do banco de dados '{db_path}'."
    )


# Exemplo de uso:
fetch_and_store_universities("China")

Foram encontradas 398 universidades em China.
Dados salvos com sucesso na tabela "China" do banco de dados './data/universities.db'.
